In [1]:
import os
import pandas as pd
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import context_precision

# 🔑 Set your API key if it's not already in your system environment variables
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "<Enter your APIs>"

print("🔄 Step 1: Initializing RAGAS Evaluator Judgements...")
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Master Baseline Query & Ground Truth Reference from our slide manual
user_query = "What is the name of our company's chief executive officer?"
reference_truth = "The CEO is Jane Smith."

print("📦 Step 2: Ingesting Logs for Configuration 1 (Basic Vector Search)...")
# Simulation 1: The correct chunk was found, but it got buried at Rank 3 behind noise!
basic_search_logs = {
    "user_input": [user_query],
    "contexts": [[
        "Doc_04.pdf: Our offices are located globally with headquarters in New York.",
        "Doc_08.pdf: Standard working hours are 9 AM to 5 PM across all departments.",
        "Doc_12.pdf (Page 4): The CEO is Jane Smith." # <-- Target chunk buried at the bottom!
    ]],
    "response": ["The CEO of the company is Jane Smith."],
    "reference": [reference_truth]
}

print("📦 Step 3: Ingesting Logs for Configuration 2 (Search + Cohere Re-ranker)...")
# Simulation 2: The Re-ranker analyzed the results and successfully pushed the target chunk to Rank 1!
reranked_search_logs = {
    "user_input": [user_query],
    "contexts": [[
        "Doc_12.pdf (Page 4): The CEO is Jane Smith.", # <-- Re-ranked directly to the top slot!
        "Doc_04.pdf: Our offices are located globally with headquarters in New York.",
        "Doc_08.pdf: Standard working hours are 9 AM to 5 PM across all departments."
    ]],
    "response": ["The CEO is Jane Smith."],
    "reference": [reference_truth]
}

# Format raw logs into HuggingFace Datasets for processing
dataset_basic = Dataset.from_dict(basic_search_logs)
dataset_reranked = Dataset.from_dict(reranked_search_logs)

print("📊 Step 4: Assessing Context Precision for Basic Search...")
report_basic = evaluate(dataset=dataset_basic, metrics=[context_precision], llm=evaluator_llm, embeddings=evaluator_embeddings)
df_basic = report_basic.to_pandas()

print("📊 Step 5: Assessing Context Precision for Re-ranked Search...")
report_reranked = evaluate(dataset=dataset_reranked, metrics=[context_precision], llm=evaluator_llm, embeddings=evaluator_embeddings)
df_reranked = report_reranked.to_pandas()

print("\n=========== 🏁 RE-RANKING PERFORMANCE COMPARISON ===========\n")
print(f"🔍 BASIC SEARCH:     Context Precision Score: {df_basic['context_precision'].values[0]:.2f}")
print(f"🚀 RE-RANKED SEARCH: Context Precision Score: {df_reranked['context_precision'].values[0]:.2f}\n")
print("============================================================\n")

🔄 Step 1: Initializing RAGAS Evaluator Judgements...
📦 Step 2: Ingesting Logs for Configuration 1 (Basic Vector Search)...
📦 Step 3: Ingesting Logs for Configuration 2 (Search + Cohere Re-ranker)...
📊 Step 4: Assessing Context Precision for Basic Search...


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

📊 Step 5: Assessing Context Precision for Re-ranked Search...


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]


=========== 🏁 RE-RANKING PERFORMANCE COMPARISON ===========

🔍 BASIC SEARCH:     Context Precision Score: 0.33
🚀 RE-RANKED SEARCH: Context Precision Score: 1.00


